In [39]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI #to load any llm from google provider

from langchain_community.tools import TavilySearchResults  #to perform internet search operationa

from langchain.tools import tool #to create tools of our own
import requests #to perform http request



In [40]:
from langchain.agents import create_react_agent, AgentExecutor

In [ ]:
#====================================
# Load Environment Variables
#====================================

os.environ["SSL_CERT_FILE"] = certifi.where() # to avoid ssl error and to resolve the path related issue
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)')))"))
Content-Length: None
API Key: 
Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in cert

In [38]:
search_tool = TavilySearchResults(max_results=2) # to allow agent to perform internet search operation with max results of 2

In [ ]:
@tool #to create tools of our own
def get_weather_data(city: str) -> str:
    """
    Fetch current weather data for a given city.

    """

    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"could not find weather data for {city}"

    return (
        f"City : {city}\n"
        f"Temperature : {data['current']['temperature']}\n"
        f"Weather : {data['current']['weather_descriptions'][0]}\n"
        f"Humidity : {data['current']['humidity']}%"
    )


In [ ]:
# checking search
result = search_tool.invoke("what is the capital of japan?")
result

In [ ]:
#====================================
# Load LLM
#====================================
llm = ChatGoogleGenerativeAI(
    model="models/gemini-3.6-flash",
    temperature=0,
    api_key=os.environ.get("GEMINI_API_KEY")
)

In [ ]:
response = llm.invoke("tell me a joke")
print(response.content)

In [ ]:
#============================
# Load prompt from langchain hub
#============================

from langsmith import Client

client=Client()


prompt = client.pull_prompt(
    "hwchase17/react",

    dangerously_pull_public_prompt=True
    )

prompt


In [ ]:
# create tools
tools = [search_tool, get_weather_data]

In [ ]:
#======================
# Create agent
#======================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
#======================
# Run agent Executor 
#======================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, #to get logs
    handle_parsing_errors=True
)

In [ ]:
response = agent_executor.invoke({
    "input":(
        "Find the current weather in Tokyo"
    )
})
print(response)